# 去重版 Stage1 0bp：风险资产美元约束归因

只读保存的 C 自有状态目标／风险上限／现金归因，区分原模型风险资产意图与 beta 上限，不改投影。

In [1]:
import hashlib,json,os,pathlib
import numpy as np,pandas as pd,qlib
from qlib.workflow import R
root=pathlib.Path.cwd();card=root/'research/stage1_dedup_0bp_projection_constraint_attribution_20261023';plan=json.loads((card/'plan.json').read_text());src=plan['sources'];sha=lambda p:hashlib.sha256(pathlib.Path(p).read_bytes()).hexdigest()
a=root/src['dedup_account_card'];b=root/src['attribution_card'];asum=json.loads((a/'summary.json').read_text());bsum=json.loads((b/'summary.json').read_text());assert sha(a/'summary.json')==src['dedup_summary_sha256'] and sha(b/'summary.json')==src['attribution_summary_sha256']
assert asum['recorder_id']==src['dedup_recorder_id'] and asum['experiment_id']==src['dedup_experiment_id'] and asum['NAS_evidence']['sha256']==src['dedup_NAS_sha256']
assert sha(root/src['original_A_projection_context'])==src['original_A_projection_context_sha256']
p=a/'qlib_recorder'/src['dedup_experiment_id']/src['dedup_recorder_id']/'artifacts';assert sha(p/'targets/C_common_dedup')==src['raw_target_artifact_sha256'] and sha(p/'paths/C_common_dedup')==src['own_path_artifact_sha256']
q=b/'qlib_recorder'/bsum['experiment_id']/bsum['recorder_id']/'artifacts/analysis/daily';assert sha(q)==src['daily_artifact_sha256']
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(root/'data/qlib/stage1_domain_heads_20261018'),region='us',kernels=1)
with R.uri_context(str(a/'qlib_recorder')):
 rec=R.get_recorder(experiment_id=src['dedup_experiment_id'],recorder_id=src['dedup_recorder_id']);assert rec.status=='FINISHED';raw=rec.load_object('targets/C_common_dedup');path=rec.load_object('paths/C_common_dedup')
with R.uri_context(str(b/'qlib_recorder')):
 rec=R.get_recorder(experiment_id=bsum['experiment_id'],recorder_id=bsum['recorder_id']);assert rec.status=='FINISHED';daily=rec.load_object('analysis/daily')['C_common_dedup']
replay=path['replay'];targets=path['targets'];assert len(raw)==len(replay)==len(targets)==len(daily)==944
pd.testing.assert_index_equal(raw.index,pd.DatetimeIndex(replay.signal_date),check_names=False)
pd.testing.assert_frame_equal(replay[['signal_date','trade_date','exit_date']].reset_index(drop=True),daily[['signal_date','trade_date','exit_date']].reset_index(drop=True))
assert np.max(np.abs(daily[['cash_opportunity_log','SOXL_selection_log','direct_fees_log','timing_and_benchmark_residual_log']].sum(axis=1)-daily.paired_log_excess))<1e-11
raw_soxl=raw.soxl_weight.to_numpy(float);raw_soxx=raw.soxx_weight.to_numpy(float);raw_risk=raw_soxl+raw_soxx;raw_beta=3*raw_soxl+raw_soxx
cap=replay.current_beta_cap.to_numpy(float);target_soxl=targets.soxl_weight.to_numpy(float);target_soxx=targets.soxx_weight.to_numpy(float);target_cash=targets.cash_weight.to_numpy(float);target_risk=target_soxl+target_soxx;target_beta=3*target_soxl+target_soxx
ceiling=np.minimum.reduce([raw_risk,cap,np.full(944,.85)]);slack=ceiling-target_risk;execute=replay.execute_open_target.to_numpy(bool);assert np.array_equal(execute,targets.execute_open_target.to_numpy(bool))
assert np.isfinite(np.c_[raw_risk,raw_beta,cap,target_risk,target_beta,target_cash]).all() and np.max(np.abs(target_soxl+target_soxx+target_cash-1))<1e-9 and np.min(target_cash)>=.15-1e-9 and np.max(target_beta-cap)<1e-8 and np.min(slack[execute])>-1e-9
binding=raw_beta>cap+1e-8;eps=1e-9;strict_raw=(raw_risk+eps<cap)&(raw_risk+eps<.85);strict_cap=(cap+eps<raw_risk)&(cap+eps<.85);strict_floor=(.85+eps<raw_risk)&(.85+eps<cap);category=np.where(strict_raw,'raw_risk',np.where(strict_cap,'beta_cap',np.where(strict_floor,'cash_floor','tie')))
rows=[]
for kind in ('raw_risk','beta_cap','cash_floor','tie'):
 mask=execute&(category==kind);rows.append({'constraint':kind,'EXECUTE_sessions':int(mask.sum()),'raw_beta_binding_sessions':int((mask&binding).sum()),'cash_opportunity_log':float(daily.loc[mask,'cash_opportunity_log'].sum()),'mean_target_cash':float(target_cash[mask].mean()) if mask.any() else None})
def periods(freq):
 out=[];labels=daily.trade_date.dt.to_period(freq)
 for period,group in daily.groupby(labels,sort=True):
  ids=group.index.to_numpy();mask=execute[ids];c=category[ids]
  out.append({'period':str(period),'sessions':len(ids),'EXECUTE_sessions':int(mask.sum()),'EXECUTE_raw_risk_limited':int((mask&(c=='raw_risk')).sum()),'EXECUTE_beta_cap_limited':int((mask&(c=='beta_cap')).sum()),'EXECUTE_cash_floor_limited':int((mask&(c=='cash_floor')).sum()),'EXECUTE_ties':int((mask&(c=='tie')).sum()),'cash_opportunity_log':float(group.cash_opportunity_log.sum()),'relative_log_excess':float(group.paired_log_excess.sum())})
 return out
window=126;roll=pd.DataFrame({'relative_log_excess':daily.paired_log_excess.rolling(window).sum().iloc[window-1:].to_numpy()});assert len(roll)==819
for kind in ('raw_risk','beta_cap','cash_floor','tie'):
 part=daily.cash_opportunity_log.where(execute&(category==kind),0.0).rolling(window).sum().iloc[window-1:].reset_index(drop=True);roll[kind+'_EXECUTE_cash_log']=part
nonwin=roll.relative_log_excess.le(0);window_cash={kind:{'nonwinning_negative_component_windows':int((nonwin&roll[kind+'_EXECUTE_cash_log'].lt(0)).sum()),'nonwinning_mean_component_log':float(roll.loc[nonwin,kind+'_EXECUTE_cash_log'].mean())} for kind in ('raw_risk','beta_cap','cash_floor','tie')}
original=json.loads((root/src['original_A_projection_context']).read_text())
result={'decision':'read_only_dedup_C_risky_dollar_ceiling_binding_layer_attributed_no_policy','plan_sha256':sha(card/'plan.json'),'source_recorder_id':src['dedup_recorder_id'],'sessions':944,'EXECUTE_sessions':int(execute.sum()),'HOLD_sessions':int((~execute).sum()),'raw_beta_binding_sessions':int(binding.sum()),'maximum_abs_EXECUTE_risky_dollar_slack':float(np.max(np.abs(slack[execute]))),'positive_slack_EXECUTE_sessions':int((execute&(slack>1e-8)).sum()),'constraint_rows':rows,'annual':periods('Y'),'quarters':periods('Q'),'rolling126_windows':819,'C_positive_windows':int((roll.relative_log_excess>0).sum()),'nonwinning_window_cash_components':window_cash,'original_A_context':{'EXECUTE_sessions':original['EXECUTE_sessions'],'constraint_rows':original['constraint_rows']},'new_market_requests':0,'new_model_fits':0,'new_native_accounts':0,'new_bootstraps':0,'new_Recorders':0}
(card/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n');print(json.dumps({k:v for k,v in result.items() if k not in ('annual','quarters','original_A_context')},ensure_ascii=False,indent=2))


{
  "decision": "read_only_dedup_C_risky_dollar_ceiling_binding_layer_attributed_no_policy",
  "plan_sha256": "55f37d233fd9d3644143bf05722abf3e7d189a33a94acc80afdbf11a3d5c767e",
  "source_recorder_id": "a42f11bdd1484bde86d4dbb1e4855af3",
  "sessions": 944,
  "EXECUTE_sessions": 822,
  "HOLD_sessions": 122,
  "raw_beta_binding_sessions": 339,
  "maximum_abs_EXECUTE_risky_dollar_slack": 1.1102230246251565e-16,
  "positive_slack_EXECUTE_sessions": 0,
  "constraint_rows": [
    {
      "constraint": "raw_risk",
      "EXECUTE_sessions": 578,
      "raw_beta_binding_sessions": 65,
      "cash_opportunity_log": -0.18155405742935204,
      "mean_target_cash": 0.293619928800824
    },
    {
      "constraint": "beta_cap",
      "EXECUTE_sessions": 244,
      "raw_beta_binding_sessions": 244,
      "cash_opportunity_log": -0.24828790405494816,
      "mean_target_cash": 0.6242444766293875
    },
    {
      "constraint": "cash_floor",
      "EXECUTE_sessions": 0,
      "raw_beta_binding_sessions

In [2]:
import hashlib,json,os,pathlib
import numpy as np,pandas as pd,qlib
from qlib.workflow import R
root=pathlib.Path.cwd();card=root/'research/stage1_dedup_0bp_projection_constraint_attribution_20261023';repair=json.loads((card/'period_cash_component_repair_plan.json').read_text());prior=json.loads((card/'result.json').read_text());sha=lambda p:hashlib.sha256(pathlib.Path(p).read_bytes()).hexdigest()
assert sha(card/'plan.json')==repair['original_plan_sha256'] and sha(card/'result.json')==repair['original_result_sha256']
base=root/'research/stage1_common_return_dedup_0bp_20261023';attr=root/'research/stage1_common_return_dedup_0bp_failure_attribution_20261023';os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(root/'data/qlib/stage1_domain_heads_20261018'),region='us',kernels=1)
with R.uri_context(str(base/'qlib_recorder')):
 rec=R.get_recorder(experiment_id='393525215581674229',recorder_id='a42f11bdd1484bde86d4dbb1e4855af3');assert rec.status=='FINISHED';raw=rec.load_object('targets/C_common_dedup');path=rec.load_object('paths/C_common_dedup')
with R.uri_context(str(attr/'qlib_recorder')):
 rec=R.get_recorder(experiment_id='126246128167380803',recorder_id='a10c938967a2433cbf80de74d924f4c1');assert rec.status=='FINISHED';daily=rec.load_object('analysis/daily')['C_common_dedup']
assert len(raw)==len(path['replay'])==len(daily)==944
raw_risk=(raw.soxl_weight+raw.soxx_weight).to_numpy(float);cap=path['replay'].current_beta_cap.to_numpy(float);execute=path['replay'].execute_open_target.to_numpy(bool);eps=1e-9
strict_raw=(raw_risk+eps<cap)&(raw_risk+eps<.85);strict_cap=(cap+eps<raw_risk)&(cap+eps<.85);strict_floor=(.85+eps<raw_risk)&(.85+eps<cap);category=np.where(strict_raw,'raw_risk',np.where(strict_cap,'beta_cap',np.where(strict_floor,'cash_floor','tie')))
report={}
for frequency in ('Y','Q'):
 rows=[];labels=daily.trade_date.dt.to_period(frequency)
 for period,group in daily.groupby(labels,sort=True):
  ids=group.index.to_numpy();components={key:float(group.loc[execute[ids]&(category[ids]==key),'cash_opportunity_log'].sum()) for key in ('raw_risk','beta_cap','cash_floor','tie')};components['HOLD']=float(group.loc[~execute[ids],'cash_opportunity_log'].sum());total=float(group.cash_opportunity_log.sum());assert abs(sum(components.values())-total)<1e-11
  rows.append({'period':str(period),'sessions':len(ids),'total_cash_opportunity_log':total,'components':components})
 report['annual' if frequency=='Y' else 'quarters']=rows
assert len(report['annual'])==4 and len(report['quarters'])==16
for old,new in zip(prior['annual'],report['annual']):assert old['period']==new['period'] and abs(old['cash_opportunity_log']-new['total_cash_opportunity_log'])<1e-11
for old,new in zip(prior['quarters'],report['quarters']):assert old['period']==new['period'] and abs(old['cash_opportunity_log']-new['total_cash_opportunity_log'])<1e-11
report.update(repair_plan_sha256=sha(card/'period_cash_component_repair_plan.json'),source_plan_sha256=prior['plan_sha256'],Qlib_Recorder_object_loads=3,model_fits=0,native_accounts=0,bootstraps=0)
(card/'period_cash_component_result.json').write_text(json.dumps(report,ensure_ascii=False,indent=2)+'\n');print(json.dumps({'annual':report['annual'],'quarter_count':len(report['quarters'])},ensure_ascii=False,indent=2))


{
  "annual": [
    {
      "period": "2023",
      "sessions": 249,
      "total_cash_opportunity_log": -0.16143570195264337,
      "components": {
        "raw_risk": -0.0634528883747974,
        "beta_cap": -0.09809811969708507,
        "cash_floor": 0.0,
        "tie": 0.0,
        "HOLD": 0.00011530611923911763
      }
    },
    {
      "period": "2024",
      "sessions": 252,
      "total_cash_opportunity_log": 0.05192570942291781,
      "components": {
        "raw_risk": -0.034122288529302776,
        "beta_cap": 0.08025041607322388,
        "cash_floor": 0.0,
        "tie": 0.0,
        "HOLD": 0.005797581878996755
      }
    },
    {
      "period": "2025",
      "sessions": 250,
      "total_cash_opportunity_log": -0.1055552432424009,
      "components": {
        "raw_risk": 0.016652549513357352,
        "beta_cap": -0.16493199758834393,
        "cash_floor": 0.0,
        "tie": 0.0,
        "HOLD": 0.04272420483258562
      }
    },
    {
      "period": "2026",
      "s

## 中文结论

C自有账户822个EXECUTE日风险美元投影全部用满；578日原风险资产意图限制现金−0.18155，244日beta上限限制现金−0.24829。2024上限现金+0.08025保护，2025却−0.16493拖累；原意图与HOLD在2025合计为正。0新增账户或拟合；这些是事后路径归因，不能据年份放宽上限。